# Shape Classification

Evaluates the rule-based 7-class shape taxonomy (Circular, Elliptical, Flattened,
Irregular, Tear-drop, Triangular, Multi-polar) using Kruskal–Wallis tests and
intraclass correlation to assess within-individual consistency.


In [ ]:
import matplotlib.patches as mpatches


## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kruskal

DATA_DIR = Path('../data')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

df      = pd.read_csv(DATA_DIR / 'shape_analysis_results.csv')
summary = pd.read_csv(DATA_DIR / 'shape_summary.csv')
print(df['shape_class'].value_counts())


## Classification rule tree

Rules applied in priority order (first matching rule wins):

| Priority | Class | Conditions |
|---|---|---|
| 1 | Multi-polar | solidity < 0.82 **AND** n_radial_peaks > 3 |
| 2 | Triangular | n_radial_peaks == 3 **AND** solidity < 0.95 **AND** circularity < 0.78 |
| 3 | Flattened | aspect_ratio ≥ 2.5 **OR** (aspect_ratio > 2.0 **AND** eccentricity > 0.88) |
| 4 | Irregular | circularity < 0.60 |
| 5 | Tear-drop | asymmetry_index > 0.15 **AND** 0.55 ≤ circularity < 0.80 **AND** 1.1 ≤ aspect_ratio < 2.0 |
| 6 | Circular | circularity > 0.85 **AND** aspect_ratio < 1.3 |
| 7 | Elliptical | circularity ≥ 0.72 **AND** 1.3 ≤ aspect_ratio < 2.5 |
| — | Irregular | fallback |


## Per-class summary statistics

In [ ]:
print(summary[['shape_class','count','pct','mean_circularity','mean_aspect_ratio',
               'mean_eccentricity','mean_efd_deviation']].to_string(index=False))


## Kruskal–Wallis: do classes differ in key features?

In [ ]:
features_to_test = ['circularity', 'aspect_ratio', 'eccentricity',
                    'solidity', 'radial_cv', 'efd_deviation']
rows = []
for feat in features_to_test:
    groups = [g[feat].dropna().values for _, g in df.groupby('shape_class')]
    groups = [g for g in groups if len(g) > 0]
    h, p = kruskal(*groups)
    rows.append({'feature': feat, 'H': round(h, 2), 'p': f'{p:.2e}', 'sig': p < 0.001})
pd.DataFrame(rows)


## Feature distributions per class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
colors = {'Circular':'#4C72B0','Elliptical':'#DD8452','Flattened':'#55A868',
          'Irregular':'#C44E52','Tear-drop':'#8172B2','Triangular':'#937860','Multi-polar':'#DA8BC3'}
feat_label = {'circularity':'Circularity','aspect_ratio':'Aspect ratio',
              'eccentricity':'Eccentricity','solidity':'Solidity',
              'radial_cv':'Radial CV','efd_deviation':'EFD deviation'}

for ax, feat in zip(axes.flat, features_to_test):
    for cls, grp in df.groupby('shape_class'):
        ax.hist(grp[feat].dropna(), bins=25, alpha=0.5, label=cls,
                color=colors.get(cls, 'grey'), density=True)
    ax.set_xlabel(feat_label.get(feat, feat))
    ax.set_ylabel('Density')

handles = [mpatches.Patch(color=v, label=k) for k, v in colors.items() if k in df['shape_class'].unique()]
fig.legend(handles=handles, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.04))
plt.suptitle('Feature distributions by shape class', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()
